retriever is a component that fetches relevant information from a knowledge source (like a database, vector store, or document collection) based on a user’s query.


Retriever is a standarization of semantic search.
Retriever is flexible and simply adjustable with chains and ragpipelines.

Wikipedia Retriever


In [7]:
from langchain_community.retrievers import WikipediaRetriever

In [12]:
# initialize the retiever
retriever = WikipediaRetriever(top_k_results=1,lang='en')

In [ ]:
# Define your querry
querry = 'the geopolitical history of india and pakistan from the perspective of a chinese'

#  get relevant wikipedia documents
docs = retriever.invoke(querry)

# print retrieved content
for i,doc in enumerate(docs):
    print(f'\n --- Result {i+1} ---')
    print(f'content:\n {doc.page_content}.....')
    print(f'content:\n {doc.metadata}.....')




 --- Result 1 ---
content:
 {'title': 'India–Pakistan war of 1965', 'summary': "The India–Pakistan war of 1965, also known as  the second Kashmir war, was an armed conflict between Pakistan and India that took place from August 1965 to September 1965. The conflict began following Pakistan's unsuccessful Operation Gibraltar, which was designed to infiltrate forces into Jammu and Kashmir to precipitate an insurgency against Indian rule. The seventeen day war caused thousands of casualties on both sides and witnessed the largest engagement of armoured vehicles and the largest tank battle since World War II. Hostilities between the two countries ended after a ceasefire was declared through UNSC Resolution 211 following a diplomatic intervention by the Soviet Union and the United States, and the subsequent issuance of the Tashkent Declaration. Much of the war was fought by the countries' land forces in Kashmir and along the border between India and Pakistan. This war saw the largest amassi

Vector Store Retriever

In [18]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [31]:
#  Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]
embedding_model = HuggingFaceEmbeddings(model_name ="sentence-transformers/all-MiniLM-L6-v2")

# Step 3: Create Chroma vector store in memory and direct adding of documents during creation
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collectionv2",
    ids=[str(i) for i in range(len(documents))] # for generating non duplicate answer

)

#  convert vectorstore into a retriever

retriever=vectorstore.as_retriever(search_kwargs={"k":3})

# querry
querry = "What is models and used for?"
results = retriever.invoke(querry)

for i,doc in enumerate(results):
     print(f"\n--- Result {i+1} ---")
     print(doc.page_content)



--- Result 1 ---
OpenAI provides powerful embedding models.

--- Result 2 ---
LangChain helps developers build LLM applications easily.

--- Result 3 ---
Chroma is a vector database optimized for LLM-based search.


MMR

MMR is used to reduce the similar results. its supports divergence results.

In [32]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [ ]:
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(model_name ="sentence-transformers/all-MiniLM-L6-v2")

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding= embedding_model
)


In [36]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)
query = "What is langchain?"
results = retriever.invoke(query)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 2 ---
LangChain is used to build LLM based applications.

--- Result 3 ---
Embeddings are vector representations of text.


Multiquery Retriever

- Similarity Retriever is good when your query is well-phrased and matches the corpus vocabulary.
- MultiQueryRetriever is better when:
- Queries are ambiguous or broad.
- You want to maximize recall (not miss relevant docs).
- Your corpus uses varied language (synonyms, paraphrases).
multiqueryretriever breaks single query into multiple query.

In [75]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_classic.retrievers import MultiQueryRetriever


In [40]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [45]:
embedding_model = HuggingFaceEmbeddings(model_name ="sentence-transformers/all-MiniLM-L6-v2")

# creates FAISS vector store 
# not tell about Persist directory, means VDB is save in RAM after closing program data is erased.
vectorstore = FAISS.from_documents(documents=all_docs,embedding=embedding_model)

# create retriever
similarity_retriever = vectorstore.as_retriever(search_type = "similarity",search_kwargs={"k": 5})




In [71]:
# Multiquery retriever
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k":5}),
    llm = ChatGroq( model="openai/gpt-oss-20b")
)
# Query
query = 'How to improve health?'

In [74]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)
for i,doc in enumerate(similarity_results):
    print(f'{i+1} \n')
    print(doc.page_content)
print("\n --> multiquery result-->\n")    
for i,doc in enumerate( multiquery_results):
    print(f'{i+1} \n')
    print(doc.page_content)

1 

Photosynthesis enables plants to produce energy by converting sunlight.
2 

Consuming leafy greens and fruits helps detox the body and improve longevity.
3 

Drinking sufficient water throughout the day helps maintain metabolism and energy.
4 

The solar energy system in modern homes helps balance electricity demand.
5 

Mindfulness and controlled breathing lower cortisol and improve mental clarity.

 --> multiquery result-->

1 

Photosynthesis enables plants to produce energy by converting sunlight.
2 

Consuming leafy greens and fruits helps detox the body and improve longevity.
3 

Drinking sufficient water throughout the day helps maintain metabolism and energy.
4 

The solar energy system in modern homes helps balance electricity demand.
5 

Mindfulness and controlled breathing lower cortisol and improve mental clarity.
6 

The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.


Contextual Compression Retriever

Only relevant output is consider.
not any irrelevent data come in output.

In [79]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_core.documents import Document

In [76]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [78]:
embedding_model = HuggingFaceEmbeddings(model_name ="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embedding_model)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

LLMChainExtractor is a utility that uses the LLM to compress or extract relevant information from retrieved documents.
Think of it as a "smart summarizer" that reduces long text chunks into only the most relevant parts for your query.


In [ ]:
# Set up the compressor using an LLM

llm = ChatGroq(model="openai/gpt-oss-20b")
compressor = LLMChainExtractor.from_llm(llm)
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)


In [82]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
    print(doc.metadata)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.
{'source': 'Doc1'}

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
{'source': 'Doc2'}

--- Result 3 ---
Photosynthesis does not occur in animal cells.
{'source': 'Doc4'}
